In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week5-lessons2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
from pyspark.sql.functions import *

In [3]:
retail_sales_df = spark.read \
.format('parquet') \
.load('/public/trendytech/datasets/ordersparquet')

In [4]:
retail_sales_df.createOrReplaceTempView("orders")

In [5]:
spark.sql("select * from orders").show(4)

+-----------+--------------------+--------+---------------+
|customer_id|          order_date|order_id|   order_status|
+-----------+--------------------+--------+---------------+
|      11599|2013-07-25 00:00:...|       1|         CLOSED|
|        256|2013-07-25 00:00:...|       2|PENDING_PAYMENT|
|      12111|2013-07-25 00:00:...|       3|       COMPLETE|
|       8827|2013-07-25 00:00:...|       4|         CLOSED|
+-----------+--------------------+--------+---------------+
only showing top 4 rows



In [6]:
spark.sql("create database if not exists itv024128")

""


In [7]:
spark.sql("show databases").filter("namespace = 'itv024128'")

namespace
itv024128


In [8]:
spark.sql("show databases").filter("namespace like 'itv024128%'")

namespace
itv024128


In [9]:
spark.sql("show tables")

database,tableName,isTemporary
default,1htab,false
default,41group_movies,false
default,4group_movies,false
default,4tab,false
default,6_flags_simon,false
default,a,false
default,aa,false
default,abc,false
default,acid,false
default,acid1,false


In [12]:
spark.sql("use itv024128")

""


In [13]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,orders,false
itv024128,orders_ext,false
,orders,true


#### Managed table

In [14]:
spark.sql("create table if not exists itv024128.orders (order_id int, order_date string, cust_id int, order_status string)")

""


In [15]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,orders,false
itv024128,orders_ext,false
,orders,true


In [16]:
spark.sql("insert into itv024128.orders select * from orders")

""


In [17]:
spark.sql("select * from  itv024128.orders ").show(5)

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|   order_status|
+--------+--------------------+-------+---------------+
|   11599|2013-07-25 00:00:...|      1|         CLOSED|
|     256|2013-07-25 00:00:...|      2|PENDING_PAYMENT|
|   12111|2013-07-25 00:00:...|      3|       COMPLETE|
|    8827|2013-07-25 00:00:...|      4|         CLOSED|
|   11318|2013-07-25 00:00:...|      5|       COMPLETE|
+--------+--------------------+-------+---------------+
only showing top 5 rows



In [18]:
spark.sql("describe table  itv024128.orders ")

col_name,data_type,comment
order_id,int,null
order_date,string,null
cust_id,int,null
order_status,string,null


In [19]:
spark.sql("describe extended   itv024128.orders ").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                 |comment|
+----------------------------+--------------------------------------------------------------------------+-------+
|order_id                    |int                                                                       |null   |
|order_date                  |string                                                                    |null   |
|cust_id                     |int                                                                       |null   |
|order_status                |string                                                                    |null   |
|                            |                                                                          |       |
|# Detailed Table Information|                                                          

In [20]:
!hadoop fs -ls -h /user/itv024128/warehouse/itv024128.db/orders

Found 4 items
-rwxr-xr-x   3 itv024128 supergroup      2.3 M 2026-07-16 03:36 /user/itv024128/warehouse/itv024128.db/orders/part-00000-ae22ff2d-1d7f-4afd-a276-04632ecec938-c000
-rwxr-xr-x   3 itv024128 supergroup      2.3 M 2026-07-16 03:44 /user/itv024128/warehouse/itv024128.db/orders/part-00000-c3bac8c0-4637-487b-8e01-7089e0c1bd75-c000
-rwxr-xr-x   3 itv024128 supergroup    596.5 K 2026-07-16 03:36 /user/itv024128/warehouse/itv024128.db/orders/part-00001-ae22ff2d-1d7f-4afd-a276-04632ecec938-c000
-rwxr-xr-x   3 itv024128 supergroup    596.5 K 2026-07-16 03:44 /user/itv024128/warehouse/itv024128.db/orders/part-00001-c3bac8c0-4637-487b-8e01-7089e0c1bd75-c000


In [21]:
spark.sql("drop table itv024128.orders ")  ## both data and meta data is deleted as its a managed table

""


In [22]:
!hadoop fs -ls -h /user/itv024128/warehouse/itv024128.db/orders

ls: `/user/itv024128/warehouse/itv024128.db/orders': No such file or directory


#### External table

In [30]:
!hadoop fs -ls -h /user/itv024128/data/orders

Found 2 items
-rw-r--r--   3 itv024128 supergroup          0 2026-07-16 03:43 /user/itv024128/data/orders/_SUCCESS
-rw-r--r--   3 itv024128 supergroup      2.9 M 2026-07-16 03:43 /user/itv024128/data/orders/part-00000


In [41]:
spark.sql("drop table itv024128.orders_ext ")

""


In [42]:
spark.sql("create table if not exists itv024128.orders_ext (order_id int, order_date string, cust_id int, order_status string) using csv location '/user/itv024128/data/orders'")

""


In [33]:
spark.sql("select * from  itv024128.orders_ext ").show(5)

+--------+--------------------+-------+---------------+
|order_id|          order_date|cust_id|   order_status|
+--------+--------------------+-------+---------------+
|       1|2013-07-25 00:00:...|  11599|         CLOSED|
|       2|2013-07-25 00:00:...|    256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|  12111|       COMPLETE|
|       4|2013-07-25 00:00:...|   8827|         CLOSED|
|       5|2013-07-25 00:00:...|  11318|       COMPLETE|
+--------+--------------------+-------+---------------+
only showing top 5 rows



In [34]:
spark.sql("show  tables")

database,tableName,isTemporary
itv024128,orders_ext,false
,orders,true


In [35]:
spark.sql("describe extended   itv024128.orders_ext ").show(truncate=False)

+----------------------------+---------------------------------------------------------+-------+
|col_name                    |data_type                                                |comment|
+----------------------------+---------------------------------------------------------+-------+
|order_id                    |int                                                      |null   |
|order_date                  |string                                                   |null   |
|cust_id                     |int                                                      |null   |
|order_status                |string                                                   |null   |
|                            |                                                         |       |
|# Detailed Table Information|                                                         |       |
|Database                    |itv024128                                                |       |
|Table                       |

In [ ]:
spark.sql("truncate table  itv024128.orders_ext ") ### cannot truncate

In [37]:
spark.sql("insert into itv024128.orders_ext select * from orders")

""


In [ ]:
###  org.apache.hadoop.security.AccessControlException: Permission denied: user=itv024128, access=WRITE, inode="/public/trendytech/retail_db/orders":itv005857:supergroup:drwxr-xr-x

In [43]:
!hadoop fs -ls -h /user/itv024128/data/orders  ## new part files created as part of insert

Found 4 items
-rw-r--r--   3 itv024128 supergroup          0 2026-07-16 03:47 /user/itv024128/data/orders/_SUCCESS
-rw-r--r--   3 itv024128 supergroup      2.9 M 2026-07-16 03:43 /user/itv024128/data/orders/part-00000
-rw-r--r--   3 itv024128 supergroup      2.3 M 2026-07-16 03:47 /user/itv024128/data/orders/part-00000-8979dd2a-2e99-452d-9c7f-35ace252fd3a-c000.csv
-rw-r--r--   3 itv024128 supergroup    596.5 K 2026-07-16 03:47 /user/itv024128/data/orders/part-00001-8979dd2a-2e99-452d-9c7f-35ace252fd3a-c000.csv


In [44]:
## update and delete also will not work in normal spark. but in Databricks it will as part of datalake

In [45]:
spark.sql("select count(*) from orders")

count(1)
68883


In [46]:
spark.sql("select count(*) from itv024128.orders_ext")

count(1)
137766


In [ ]:
spark.sql("delete from itv024128.orders_ext where order_id =1 ") ## : DELETE is only supported with v2 tables.